# tcrkit — quick tour

| | capability | does |
|---|---|---|
| 1 | `stitch` | V/J + CDR3 → a full-length TCR chain |
| 2 | `anarci` | a protein sequence → V/J genes + CDR loops |
| 3 | `add_cdr3_junctions` | a CDR3 missing its C/F → one that will stitch |
| 4 | `normalize_tcr` / `normalize_mhc` | messy gene and allele names → IMGT |
| 5 | `mhc_sequence` | an MHC allele → its protein sequence |
| 6 | `build_tcr_pmhc` | all of the above, over a table, in one call |

Three things hold everywhere, so there is little else to remember:

- **One value or many.** The same function takes a string, a list, or a DataFrame.
- **A DataFrame comes back**, with your input echoed beside the result. `squeeze=True`
  gives the bare value instead.
- **One `error` column**, and species are always `human` / `mouse` (`Mus musculus` and
  friends are understood on input).

In [1]:
import pandas as pd

import tcrkit

---
## 1. `stitch` — one or many

One rearrangement:

In [2]:
tcrkit.stitch('TRBV19', 'TRBJ2-7', 'CASSIRSSYEQYF', chain='beta')

,v,j,cdr3,sequence,sequence_aa,error,stitchr_warnings
0,TRBV19,TRBJ2-7,CASSIRSSYEQYF,ATGAGCAACCAGGTGCTCTGCTGTGTGGTCCTTTGTTTCCTGGGAG...,MSNQVLCCVVLCFLGANTVDGGITQSPKYLFRKEGQNVTLSCEQNL...,-,No valid leader region allele determined yet f...


Many — the same function, given a table. Add a `chain` column and alpha and beta stitch
together:

In [3]:
tcrs = pd.DataFrame({'v':     ['TRBV19',        'TRAV12-2'],
                     'j':     ['TRBJ2-7',       'TRAJ23'],
                     'cdr3':  ['CASSIRSSYEQYF', 'CAVNTGGGNKLTF'],
                     'chain': ['beta',          'alpha']})

out = tcrkit.stitch(tcrs, progress_bar=False)
out[['chain', 'v', 'j', 'error']].assign(aa_len=out.sequence_aa.str.len())

,chain,v,j,error,aa_len
0,beta,TRBV19,TRBJ2-7,-,308
1,alpha,TRAV12-2,TRAJ23,-,273


Another species — `species='mouse'`, and the mouse germline data downloaded once with
`stitchrdl -s mouse`. Here is the OT-I receptor, the standard ovalbumin-specific mouse
TCR, both chains in one call:

In [4]:
mouse = tcrkit.stitch([('TRAV14-1', 'TRAJ33',  'CAASDNYQLIW'),      # OT-I alpha
                       ('TRBV12-1', 'TRBJ2-7', 'CASSRANYEQYF')],    # OT-I beta
                      chain=['alpha', 'beta'], species='mouse', progress_bar=False)

mouse[['v', 'j', 'cdr3', 'error']].assign(aa_len=mouse.sequence_aa.str.len())

,v,j,cdr3,error,aa_len
0,TRAV14-1,TRAJ33,CAASDNYQLIW,-,275
1,TRBV12-1,TRBJ2-7,CASSRANYEQYF,-,313


`species` is `'human'` or `'mouse'` here and in every other capability, and spellings
like `Mus musculus` are understood too. Like `chain`, it can be one value for the whole
input, one per row, or a column in the frame.

ANARCI agrees these are mouse genes — worth checking when you stitch a species you use
less often:

In [5]:
tcrkit.anarci(list(mouse.sequence_aa), allow={'A', 'B'}, progress_bar=False)[
    ['species', 'chain_type', 'v_gene', 'j_gene', 'CDR3_aa']]

,species,chain_type,v_gene,j_gene,CDR3_aa
0,mouse,A,TRAV14-1*01,TRAJ33*01,AASDNYQLI
1,mouse,B,TRBV12-1*01,TRBJ2-7*01,ASSRANYEQY


If your columns are named differently, say so with `col_mapping` — tcrkit never guesses
your schema:

```python
tcrkit.stitch(df, col_mapping={'v': 'v_call', 'j': 'j_call', 'cdr3': 'junction_aa'},
              chain='beta')
```

A row that fails says why in `error` instead of raising, so one bad row never costs you
the table.

---
## 2. `anarci` — annotate, and **always pass `allow`**

ANARCI scores against every chain type it knows, and the alpha and delta loci overlap.
Unrestricted it will quietly call an alpha chain a delta chain and hand you a TRDJ gene:

In [6]:
alpha = ('AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMIFLIYQGSYDQQNATEGRYSLNFQKARKSANL'
         'VISASQLGDSAMYFCAMSQLNSGNTPLVFGKGTRLSVIA')

pd.concat([tcrkit.anarci(alpha).assign(passed='nothing'),
           tcrkit.anarci(alpha, allow={'A', 'B'}).assign(passed="allow={'A','B'}")],
          ignore_index=True)[['passed', 'chain_type', 'v_gene', 'j_gene', 'CDR3_aa']]

,passed,chain_type,v_gene,j_gene,CDR3_aa
0,nothing,D,TRAV14/DV4*01,TRDJ4*01,AMSQLNSGNTPLV
1,"allow={'A','B'}",A,TRAV14/DV4*01,TRAJ29*01,AMSQLNSGNTPLV


Same sequence, but `TRDJ4*01` versus `TRAJ29*01` — and nothing warns you. **Use
`allow={'A', 'B'}` for alpha/beta TCRs**, or `{'G', 'D'}` for gamma/delta.

Many sequences at once; a dict names them and those names become the index:

In [7]:
beta = ('MNAGVTQTPKFRVLKTGQSMTLLCAQDMNHDYMYWYRQDPGMGLRLIHYSVGEGTTAKGEVPDGYNVSRLKKQNFLL'
        'GLESAAPSQTSVYFCASSFTDTQYFGPGTRLTVL')

tcrkit.anarci({'a': alpha, 'b': beta, 'junk': 'NOTASEQUENCE'},
              allow={'A', 'B'}, progress_bar=False)

,sequence,CDR1_aa,CDR1_aa_start,CDR1_aa_end,CDR2_aa,CDR2_aa_start,CDR2_aa_end,CDR3_aa,CDR3_aa_start,CDR3_aa_end,query_start,query_end,species,chain_type,v_gene,v_identity,j_gene,j_identity,error
a,AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMI...,TSDPSYG,27,33,QGSYDQQN,51,58,AMSQLNSGNTPLV,93,105,0,115,human,A,TRAV14/DV4*01,0.968750,TRAJ29*01,0.928571,-
b,MNAGVTQTPKFRVLKTGQSMTLLCAQDMNHDYMYWYRQDPGMGLRL...,MNHDY,28,32,SVGEGT,50,55,ASSFTDTQY,93,101,1,111,human,B,TRBV6-2*01,0.978947,TRBJ2-3*01,1.000000,-
junk,NOTASEQUENCE,-,-1,-1,-,-1,-1,-,-1,-1,-1,-1,-,-,-,-1.000000,-,-1.000000,KeyError: '27'


Every column above is a summary. The numbering underneath it — which residue sits at
which IMGT position — comes back with `include_imgt_index=True`, as a
position&nbsp;&rarr;&nbsp;residue dict you can lay out as rows:

In [8]:
num = tcrkit.anarci(alpha, allow={'A', 'B'}, include_imgt_index=True, progress_bar=False)

imgt = pd.Series(num['imgt_dict'].iloc[0], name='residue').rename_axis('imgt_position')
imgt.to_frame().T

imgt_position,1,2,3,4,5,6,7,8,9,10,...,118,119,120,121,122,123,124,125,126,127
residue,A,Q,K,I,T,Q,T,Q,P,G,...,F,G,K,G,T,R,L,S,V,I


The point of IMGT numbering is that the positions mean the same thing in every chain,
so several sequences can share one set of columns. `separate_index=True` lays them out
that way — one column per position, plus `_position` / `_suffix` header rows so you can
slice the IMGT axis numerically. Here is the junction, 104&ndash;118:

In [9]:
wide = tcrkit.anarci({'alpha': alpha, 'beta': beta}, allow={'A', 'B'},
                     separate_index=True, progress_bar=False)

junction = (wide.loc['_position'] >= 104) & (wide.loc['_position'] <= 118)
wide.loc[['_position', 'alpha', 'beta'], junction]

,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118
_position,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118
alpha,C,A,M,S,Q,L,N,S,G,N,T,P,L,V,F
beta,C,A,S,S,F,T,-,-,-,-,D,T,Q,Y,F


Same columns, so the two chains line up: `C` at 104 and `F` at 118 in both, and gaps
(`-`) where the shorter beta loop has nothing to put. `CDR3_aa` is IMGT 105&ndash;117, so
it stops *inside* those anchors and carries no C104/F118 — which is what section 3 is
about. No `hmmscan` binary is needed; tcrkit falls back to an in-process pyhmmer backend.

---
## 3. `add_cdr3_junctions` — when a CDR3 will not stitch

Stitchr needs the CDR3 to carry its IMGT junction residues. Many sources report the
105–117 core instead, which fails outright:

In [10]:
tcrkit.stitch('TRBV19', 'TRBJ2-7', 'ASSIRSSYEQY', chain='beta')[['cdr3', 'error']]

,cdr3,error
0,ASSIRSSYEQY,Unable to locate C terminus of CDR3 in J gene ...


`stitch` repairs nothing — that is a separate step first. The J gene decides the 118
residue, and it is not always F:

In [11]:
tcrkit.add_cdr3_junctions(cdr3=['ASSIRSSYEQY', 'AVMDSNYQLI', 'AVRD'],
                          j=['TRBJ2-7', 'TRAJ33', 'TRAJ35'])[
    ['cdr3', 'j', 'cdr3_fixed', 'junction', 'error']]

,cdr3,j,cdr3_fixed,junction,error
0,ASSIRSSYEQY,TRBJ2-7,CASSIRSSYEQYF,C/F,None
1,AVMDSNYQLI,TRAJ33,CAVMDSNYQLIW,C/W,None
2,AVRD,TRAJ35,CAVRDC,C/C,None


F, W and C — which is why this is table-driven rather than a blind `C…F`. Over a frame it
is two lines, repair then stitch:

In [12]:
df = pd.DataFrame({'v': ['TRBV19', 'TRAV1-2'], 'j': ['TRBJ2-7', 'TRAJ33'],
                   'cdr3': ['ASSIRSSYEQY', 'AVMDSNYQLI'], 'chain': ['beta', 'alpha']})

df['cdr3'] = tcrkit.add_cdr3_junctions(df)['cdr3_fixed']     # step 1
tcrkit.stitch(df, progress_bar=False)[['chain', 'cdr3', 'error']]   # step 2

,chain,cdr3,error
0,beta,CASSIRSSYEQYF,-
1,alpha,CAVMDSNYQLIW,-


---
## 4. Standardizing names — `normalize_tcr` and `normalize_mhc`

Gene symbols to IMGT. `species` is optional: left out, it tries human then mouse and
tells you which won.

In [13]:
tcrkit.normalize_tcr(['TCRBV20S1', 'TRAV23/6', 'TCRAV13S1', 'TRAV14D-3/DV8', 'junk'],
                     log_failures=False)

,symbol,standardized,species,error
0,TCRBV20S1,TRBV20-1,human,NaN
1,TRAV23/6,TRAV23/DV6,human,NaN
2,TCRAV13S1,TRAV22,human,NaN
3,TRAV14D-3/DV8,TRAV14D-3/DV8,mouse,NaN
4,junk,NaN,NaN,not recognised by tidytcells


> **Watch out:** a species that is *valid but wrong* silently changes the gene —
> `normalize_tcr('TRBV20-1', species='mouse')` returns `TRBV20`, a different gene, with no
> error. An unrecognised species does raise. Leaving `species` out avoids this.

MHC/HLA allele names, any spelling:

In [14]:
tcrkit.normalize_mhc(['A*0201', 'HLA-B57:01', 'H2-Kb', 'DRB1*04:01', 'bad'],
                     errors='coerce')

,allele_input,mhc_prefix,species,gene,allele,mhc_class,error
0,A*0201,HLA,human,A,A0201,I,NaN
1,HLA-B57:01,HLA,human,B,B5701,I,NaN
2,H2-Kb,H2,mouse,K,H2-Kb,I,NaN
3,DRB1*04:01,HLA,human,DRB1,DRB1*0401,II,NaN
4,bad,NaN,NaN,NaN,NaN,NaN,ParseError: Could not parse 'bad'


`errors='coerce'` records the failure instead of raising. `paired=True` returns the
alpha/beta layout for any class — class I pairs with its invariant B2M.

---
## 5. `mhc_sequence` — an allele to its sequence

Naming and sequence are different jobs: `normalize_mhc` tidies the *name*,
`mhc_sequence` fetches the *sequence* from a reference of 251 alleles.

> That reference (`mhc_sequences.csv`) is derived from IMGT/HLA and is **not kept in git**,
> so these two cells are the only ones that need a file a fresh clone does not have. If
> they raise `FileNotFoundError`, copy it into `src/tcrkit/data/` or pass your own with
> `table=`.

In [15]:
tcrkit.mhc_sequence(['HLA-A*02:01', 'A*0201', 'H2-Kb', 'DRB1*04:01', 'A2'])[
    ['allele_input', 'allele', 'length', 'mhc_class', 'species', 'error']]

,allele_input,allele,length,mhc_class,species,error
0,HLA-A*02:01,A0201,283.0,I,human,NaN
1,A*0201,A0201,283.0,I,human,NaN
2,H2-Kb,H2-Kb,286.0,I,mouse,NaN
3,DRB1*04:01,DRB1*0401,198.0,II,human,NaN
4,A2,A2,NaN,I,human,'A2' not in the reference


The first two are the same allele spelled differently and find the same row. `A2` is a
serotype, not an allele, so it misses — reported, not raised.

`full=True` adds the transmembrane and cytoplasmic tails; `paired=True` returns both
chains of the heterodimer:

In [16]:
pair = tcrkit.mhc_sequence(['A*0201', 'DRB1*04:01'], paired=True)
pair[['allele_input', 'mhc_a_allele', 'mhc_b_allele', 'mhc_class']].assign(
    a_len=pair.mhc_a_sequence.str.len(), b_len=pair.mhc_b_sequence.str.len())

,allele_input,mhc_a_allele,mhc_b_allele,mhc_class,a_len,b_len
0,A*0201,A0201,B2M_human,I,283,100
1,DRB1*04:01,DRA0101,DRB1*0401,II,192,198


---
## 6. `build_tcr_pmhc` — the whole complex in one call

Everything above, over a table: normalize the calls, complete the junctions, stitch both
chains, fetch both MHC sequences, annotate with ANARCI, and check itself. Columns you
brought along (an epitope, an id) come through untouched.

In [17]:
raw = pd.DataFrame([
    {'id': 'a', 'epitope': 'GILGFVFTL',
     'alpha_v': 'TRAV1-2', 'alpha_j': 'TRAJ33',  'alpha_cdr3': 'AVMDSNYQLI',
     'beta_v': 'TCRBV20S1', 'beta_j': 'TRBJ2-7', 'beta_cdr3': 'ASSLGQAYEQY',
     'mhc_a': 'HLA-A*02:01'},
    {'id': 'b', 'epitope': 'BROKEN',
     'alpha_v': 'TRAV1-2', 'alpha_j': 'TRAJ33',  'alpha_cdr3': 'AVMDSNYQLI',
     'beta_v': 'NOTAGENE', 'beta_j': 'TRBJ2-7',  'beta_cdr3': 'ASSLGQAYEQY',
     'mhc_a': 'A2'},
])

built = tcrkit.build_tcr_pmhc(raw, progress_bar=False)
built[['id', 'epitope', 'beta_v', 'beta_cdr3', 'mhc_a_allele', 'mhc_b_allele', 'ok']]

,id,epitope,beta_v,beta_cdr3,mhc_a_allele,mhc_b_allele,ok
0,a,GILGFVFTL,TRBV20-1,CASSLGQAYEQYF,A0201,B2M_human,True
1,b,BROKEN,NOTAGENE,CASSLGQAYEQYF,A2,B2M_human,False


Row `a` was cleaned and built; row `b` had a bad V gene and a serotype instead of an
allele, and says so without stopping the rest:

In [18]:
built[['id', 'ok', 'error']]

,id,ok,error
0,a,True,-
1,b,False,beta V call not recognised | beta did not stit...


It also checks itself: ANARCI re-reads each *stitched* sequence with no knowledge of the
input, and `{chain}_cdr3_ok` says whether it recovered the CDR3 that went in — if it did,
the normalizing, the junction repair and the stitching all agreed.

In [19]:
built[['id', 'beta_v', 'beta_anarci_v', 'beta_cdr3_ok', 'alpha_cdr3_ok']]

,id,beta_v,beta_anarci_v,beta_cdr3_ok,alpha_cdr3_ok
0,a,TRBV20-1,TRBV20-1*01,True,True
1,b,NOTAGENE,NaN,NaN,True


---
## Where to go next

- `tcrkit_metrics.ipynb` — diversity, clonality, overlap, OLGA, tcrdist
- `tcrkit --help` — the same six capabilities from the command line
- `help(tcrkit.stitch)` — every function lists the input shapes it takes and the columns
  it returns
- `README.md` — installation and the one post-install step `stitch` needs